# Stochastic Allen–Cahn Phase-Field Conditioning

This notebook reproduces the Allen–Cahn experiment of *LatentFlow: A General Framework for Conditioning Stochastic Processes* (Sharrock, Astfalck & Moss, 2026; Figures 5 and 19). A phase field on $[-1, 1]^2$ follows the stochastic Allen–Cahn equation

$$\partial_t u = \epsilon^2 \Delta u + \kappa\,(u - u^3) + \sigma\,\xi(\mathbf{x}, t),$$

with periodic boundaries, starting from small random noise. The double-well reaction drives $u$ towards the phases $\pm 1$, giving stochastic domain coarsening. The initial field and the noise on a $64 \times 64$ grid at 30 times give **122,880** Gaussian innovations. Guidance then conditions on

1. **sparse sensors**: noisy point observations at 36 locations, at each of 6 times;
2. **quadrant means**: the final field's mean in each quadrant is $(+0.1, -0.1, -0.1, +0.1)$, a checkerboard;
3. **an annular ring**: positive mean ($+0.3$) in the annulus $0.30 \le r \le 0.65$, negative ($-0.3$) in the core and exterior;
4. **a phase front**: means $(-0.2, 0, +0.2)$ in three vertical strips with boundaries at $|x| = 0.3$.

Conditions 2–4 constrain only statistics of the **final** field; the earlier frames follow from the dynamics.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from matplotlib.patches import Rectangle

from flow_inference.conditions import BinaryCondition, EqualityCondition
from flow_inference.flows import GaussianIncrementFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.maps import BaseMap
from flow_inference.sampler import IntegratorSampler

torch.set_default_dtype(torch.float64)

## 1. Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)

G = 64  # grid size on [-1, 1]^2
NUM_FRAMES = 30  # the initial field plus 29 noisy steps
T_FINAL = 1.0
DT = T_FINAL / (NUM_FRAMES - 1)
num_samples = 4
NUM_STEPS = 100  # flow integration steps

SNAP_TIMES = [0.0, 0.21, 0.41, 0.62, 0.83, 1.0]
SNAP_FRAMES = [round(t / DT) for t in SNAP_TIMES]
CMAP, VMIN, VMAX, FIELD_LABEL = "plasma", -1.0, 1.0, "field value"

DX = 2.0 / G
centres = torch.linspace(-1 + DX / 2, 1 - DX / 2, G, device=device)
YY, XX = torch.meshgrid(-centres, centres, indexing="ij")  # row 0 is y = 1
RADIUS = (XX**2 + YY**2).sqrt()

## 2. The Allen–Cahn simulator

`AllenCahnMap` is the generator $T_\vartheta$: explicit Euler–Maruyama with a periodic five-point Laplacian. The first latent slice sets the initial field and each later slice is one step's noise.

In [ ]:
class AllenCahnMap(BaseMap):
    """Latent (B, 1, NUM_FRAMES * G * G) -> phase fields (B, 1, NUM_FRAMES * G * G)."""

    def __init__(self, eps=0.06, rate=5.0, sigma=0.6, init_scale=0.3):
        super().__init__(latent_dim=NUM_FRAMES * G * G, output_shape=(NUM_FRAMES, G, G), device=device)
        self.eps, self.rate, self.sigma, self.init_scale = eps, rate, sigma, init_scale

    def _map(self, z):
        B = z.shape[0]
        z = z.reshape(B, NUM_FRAMES, G, G)
        u = self.init_scale * z[:, 0]
        frames = [u]
        for k in range(1, NUM_FRAMES):
            laplacian = (u.roll(1, -1) + u.roll(-1, -1) + u.roll(1, -2) + u.roll(-1, -2) - 4 * u) / DX**2
            u = u + DT * (self.eps**2 * laplacian + self.rate * (u - u**3)) + self.sigma * DT**0.5 * z[:, k]
            frames.append(u)
        return torch.stack(frames, dim=1).reshape(B, 1, -1)


flow = GaussianIncrementFlow(map=AllenCahnMap(), device=device)
sampler = IntegratorSampler(flow=flow)

## 3. Helpers

`sample` draws prior or guided fields, and `plot_snapshots` lays several regimes out as in the paper: two draws per regime, one row each, at six times.

In [ ]:
def sample(condition=None):
    """`num_samples` fields (B, num_frames, G, G): the prior if `condition` is None, else guided."""
    guidance = None if condition is None else GuidanceMC(flow, condition).to(device)
    out = sampler.sample(num_samples, guidance=guidance, num_steps=NUM_STEPS).detach()
    return out.reshape(num_samples, NUM_FRAMES, G, G)


def plot_snapshots(regimes, draws=2, overlay=None):
    """Rows: `draws` samples of each regime; columns: the fields at SNAP_TIMES.

    `regimes` is a list of (label, samples); `overlay(ax, label, frame)` draws a regime's
    condition on a panel.
    """
    rows = len(regimes) * draws
    fig, axes = plt.subplots(rows, len(SNAP_FRAMES), figsize=(1.7 * len(SNAP_FRAMES) + 1.6, 1.7 * rows),
                             squeeze=False)
    for r, (label, samples) in enumerate(regimes):
        for d in range(draws):
            row = r * draws + d
            for col, frame in enumerate(SNAP_FRAMES):
                ax = axes[row, col]
                im = ax.imshow(samples[d, frame].cpu(), cmap=CMAP, vmin=VMIN, vmax=VMAX)
                ax.set_xticks([])
                ax.set_yticks([])
                if overlay is not None:
                    overlay(ax, label, frame)
                if row == 0:
                    ax.set_title(f"t = {frame * DT:.2f}", fontsize=9)
            if d == 0:
                axes[row, 0].set_ylabel(label, fontsize=10, fontweight="bold")
    fig.colorbar(im, ax=axes, fraction=0.02, pad=0.01, label=FIELD_LABEL)
    return fig


def to_pixels(x, y):
    """Domain coordinates on [-1, 1]^2 -> image (column, row)."""
    return (x + 1) / DX - 0.5, (1 - y) / DX - 0.5

## 4. Prior

Small random initial noise coarsens into $\pm 1$ domains at random positions.

In [ ]:
prior_samples = sample()
plot_snapshots([("Prior", prior_samples)]);

## 5. Sparse sensors

Take one prior field as the "truth" and observe it on a $6 \times 6$ grid of sensors at each snapshot time, with noise of standard deviation $0.05$. `SensorObservation` is an `EqualityCondition` whose `sigma` is that noise level.

In [ ]:
truth = sample()[0]
sensor_idx = torch.linspace(5, G - 6, 6, device=device).round().long()
sensor_rows, sensor_cols = torch.meshgrid(sensor_idx, sensor_idx, indexing="ij")
sensors = (sensor_rows * G + sensor_cols).reshape(-1)  # 36 flat pixel indices
sensor_frames = torch.tensor(SNAP_FRAMES, device=device)
OBS_NOISE = 1e-5
generator = torch.Generator(device=device).manual_seed(SEED)
observations = truth.reshape(NUM_FRAMES, -1)[sensor_frames][:, sensors]
observations = observations + OBS_NOISE * torch.randn(observations.shape, device=device, generator=generator)


class SensorObservation(EqualityCondition):
    """Noisy point observations: the field at `pixels` in each of `frames` equals `observations`."""

    def __init__(self, frames, pixels, observations, sigma):
        super().__init__(sigma=sigma)
        self.register_buffer("frames", frames)
        self.register_buffer("pixels", pixels)
        self.register_buffer("observations", observations)  # (num_frames, num_pixels)

    def residual(self, x):
        fields = x.reshape(x.shape[0], NUM_FRAMES, -1)[:, self.frames][:, :, self.pixels]
        return (fields - self.observations[None]).reshape(x.shape[0], 1, -1)


sensor_condition = SensorObservation(sensor_frames, sensors, observations, sigma=OBS_NOISE)
sensor_samples = sample(sensor_condition)
residual = sensor_condition.residual(sensor_samples.reshape(num_samples, 1, -1)).abs().mean().item()
print(f"mean |error| at the sensors: {residual:.3f} (observation noise sd {OBS_NOISE})")


def draw_sensors(ax):
    ax.scatter(sensor_cols.reshape(-1).cpu(), sensor_rows.reshape(-1).cpu(), s=10,
               facecolors="none", edgecolors="white", linewidths=0.8)


plot_snapshots([("Truth", truth[None].expand(2, -1, -1, -1)), ("Sparse sensors", sensor_samples)],
               overlay=lambda ax, label, frame: draw_sensors(ax));

## 6. Region means of the final field

`FinalRegionMeans` is an `EqualityCondition` on the final frame: the mean over each region (a boolean mask) must equal its target. One class covers all three layouts.

In [ ]:
class FinalRegionMeans(EqualityCondition):
    """The final field's mean over each of R masks equals the matching target."""

    def __init__(self, masks, targets, sigma=1e-3):
        super().__init__(sigma=sigma)
        weights = masks.double()
        self.register_buffer("weights", weights / weights.sum(dim=(1, 2), keepdim=True))  # (R, G, G)
        self.register_buffer("targets", torch.as_tensor(targets, device=masks.device, dtype=weights.dtype))

    def residual(self, x):
        final = x.reshape(x.shape[0], NUM_FRAMES, G, G)[:, -1]
        means = torch.einsum("bij,rij->br", final, self.weights)
        return (means - self.targets[None])[:, None, :]


LAYOUTS = {
    "Quadrant means": (
        torch.stack([(XX < 0) & (YY > 0), (XX > 0) & (YY > 0), (XX < 0) & (YY < 0), (XX > 0) & (YY < 0)]),
        [0.4, -0.4, -0.4, 0.4],
    ),
    "Annular ring": (
        torch.stack([RADIUS < 0.3, (RADIUS >= 0.3) & (RADIUS <= 0.65), RADIUS > 0.65]),
        [-0.5, 0.5, -0.5],
    ),
    "Phase front": (
        torch.stack([XX < -0.3, XX.abs() <= 0.3, XX > 0.3]),
        [-0.4, 0.0, 0.4],
    ),
}
layout_samples = {}
for name, (masks, targets) in LAYOUTS.items():
    condition = FinalRegionMeans(masks, targets)
    layout_samples[name] = sample(condition)
    prior_error = condition.residual(prior_samples.reshape(num_samples, 1, -1)).abs().mean().item()
    guided_error = condition.residual(layout_samples[name].reshape(num_samples, 1, -1)).abs().mean().item()
    print(f"{name:15s} mean |region-mean error|: prior={prior_error:.3f}  guided={guided_error:.4f}")

## 7. Summary

The five regimes as in Figure 19 of the paper. Region outlines and target signs are drawn on the final frame, which is the only one constrained.

In [ ]:
def draw_layout(ax, name):
    if name == "Quadrant means":
        ax.axhline(G / 2 - 0.5, color="white", linestyle=":", linewidth=1)
        ax.axvline(G / 2 - 0.5, color="white", linestyle=":", linewidth=1)
        for (x, y), text in zip([(-0.5, 0.5), (0.5, 0.5), (-0.5, -0.5), (0.5, -0.5)], ["+", "−", "−", "+"]):
            ax.text(*to_pixels(x, y), text, color="white", ha="center", va="center", fontsize=12, fontweight="bold")
    elif name == "Annular ring":
        for radius in (0.3, 0.65):
            ax.add_patch(plt.Circle(to_pixels(0.0, 0.0), radius / DX, fill=False, color="white", linestyle=":"))
        for (x, y), text in zip([(0.0, 0.0), (0.0, 0.47), (0.82, 0.82)], ["−", "+", "−"]):
            ax.text(*to_pixels(x, y), text, color="white", ha="center", va="center", fontsize=12, fontweight="bold")
    elif name == "Phase front":
        for x in (-0.3, 0.3):
            ax.axvline(to_pixels(x, 0.0)[0], color="white", linestyle=":", linewidth=1)
        for x, text in zip((-0.65, 0.0, 0.65), ["−", "0", "+"]):
            ax.text(*to_pixels(x, 0.0), text, color="white", ha="center", va="center", fontsize=12, fontweight="bold")


def overlay(ax, label, frame):
    if label == "Sparse sensors":
        draw_sensors(ax)
    elif frame == NUM_FRAMES - 1:
        draw_layout(ax, label)


plot_snapshots([("Prior", prior_samples), ("Sparse sensors", sensor_samples)] + list(layout_samples.items()),
               overlay=overlay);